In [0]:
# ================================================================
# NOTEBOOK: nb_silver_exchange_rates
# PURPOSE:  Daily Full Refresh Bronze → Silver for Exchange Rates
# RUN:      Daily (after ADF pl_ingest_exchange_rates runs)
#           Also run ONCE manually now to seed Silver
# SOURCE:   bronze/exchange_rates/ (parquet - already flattened)
#           166 currencies × N days = N×166 rows
# TARGET:   silver/exchange_rates/ (Delta format)
# =================================================================
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

from pyspark.sql import functions as F
from pyspark.sql.functions import (trim, when, col, to_date, round, lit, upper, to_timestamp)


BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/exhanges_path/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/exhanges_path/"


# Currencies relevant to ShopSense business
# (international sellers + common tourist/business currencies)
RELEVANT_CURRENCIES = [
    "INR",  # Indian Rupee          ← most important, all orders in INR
    "USD",  # US Dollar             ← international sellers
    "EUR",  # Euro                  ← European brands
    "GBP",  # British Pound         ← UK brands
    "AED",  # UAE Dirham            ← Gulf NRI customers
    "SGD",  # Singapore Dollar      ← Southeast Asia
    "JPY",  # Japanese Yen          ← electronics from Japan
    "CNY",  # Chinese Yuan          ← manufacturing costs
    "AUD",  # Australian Dollar     ← some seller base
    "CAD",  # Canadian Dollar       ← some seller base
]


# ── READ Bronze ───────────────────────────────────────────────
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[DONE] Read rows : {bronze_df.count()} rows")
print(f"  (Expected: 166 currencies X number of days)")

# ── STEP 1: Remove nulls on key columns ──────────────────
bronze_df = ( bronze_df 
    .filter(col("base_code").isNotNull())
    .filter(col("currency").isNotNull())
    .filter(col("rate").isNotNull())
    .filter(col("rate").cast("double") > 0)
    .filter(col("time_last_update_utc").isNotNull())
)

print(f" [CLEAN] After null/zero removal: {bronze_df.count()} rows")

# ── STEP 2: Parse date from time_last_update_utc ─────────────
# Raw format: "Tue, 30 Jun 2026 00:02:31 +0000"
# We extract just the date portion for joining with orders

silver_df =(  bronze_df     
    .withColumn("rate_timestamp",  to_timestamp(col("time_last_update_utc"),
    "EEE, dd MMM yyyy HH:mm:ss Z"))
    .withColumn("rate_date",  to_date(col("rate_timestamp")))
)


# Verify parsing worked
failed_parse = silver_df.filter(col("rate_date").isNull()).count()
if failed_parse > 0:
    print(f"[WARN] {failed_parse} rows failed date parsing — check format ")
else:
    print("[OK] Date parsing successful" )


# ── STEP 3: Type casting & Standardize string columns  ──────────────────────────────────────

silver_df = (
    silver_df
    .withColumn("rate",     col("rate").cast("decimal(10,2)"))
    .withColumn("currency",   upper(trim(col("currency"))))
    .withColumn("base_code", upper(trim(col("base_code"))))
)

# ── STEP 4: Filter to relevant currencies only ────────────────
# Keeps Silver lean — no need for 166 currencies in analytics
# Gold only needs INR and a few others for enrichment

silver_df = silver_df.filter(
    col("currency").isin(RELEVANT_CURRENCIES)
)
print(f"[FILTER] After keeping relevant {len(RELEVANT_CURRENCIES)} currencies:{silver_df.count()} ")


# ── STEP 5: Business derived columns ─────────────────────────

# INR conversion rate from USD base
# If base is USD and currency is INR → this IS the INR rate
# Example: base=USD, currency=INR, rate=95.0 means 1 USD = 95.0 INR

silver_df = (
    silver_df
    .withColumn("usd_to_inr_rate",
        when((col("base_code") == "USD") & (col("currency") == "INR"), col("rate"))
        .otherwise(None)
    )


# Inverse rate: INR per 1 unit of currency
# Useful for converting product prices TO INR
# Example: 1 USD product → multiply by inr_per_unit → INR price

    # Inverse rate: INR per 1 unit of currency
    # Useful for converting product prices TO INR
    # Example: 1 USD product → multiply by inr_per_unit → INR price
    .withColumn("inr_per_unit",
        when(col("usd_to_inr_rate").isNotNull(), lit(1.0))
        .otherwise(None)   # filled in Gold via join with INR rate
    )


# Year and Month for partitioning and trend analysis
.withColumn("rate_year", F.year("rate_date"))
.withColumn("rate_month", F.month("rate_date"))


# Day of week (are rates different on weekends?)
.withColumn("rate_day_of_week",  F.dayofweek("rate_date"))
.withColumn("is_weekend",
 col("rate_day_of_week").isin([1,7]))  # 1=Sunday, 7=Saturday


# Stale flag: if rate_date is more than 2 days old
# (means API might have missed a day)
.withColumn("is_stale",   F.datediff(F.current_date(),col("rate_date")) > 2 )



# Metadata
.withColumn("_silver_load_ts",  F.current_timestamp())
.withColumn("_source",  lit("exchange_rate_api_daily"))
)

# ── STEP 6: Deduplication ─────────────────────────────────────
# In case bronze has duplicate records for same currency + date

before = silver_df.count()
silver_df = silver_df.dropDuplicates(["currency", "rate_date"])
after = silver_df.count()

if before != after:
    print(f"[WARN] Removed {before - after} duplicate currency+date rows")


# ── STEP 7: Write Silver as Delta (overwrite daily) ───────────
(
 silver_df.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.save(SILVER_PATH)
)

# ── STEP 8: Verify + Show Key Metrics ────────────────────────

total = silver_df.count()
unique_curr = silver_df.select("currency").distinct().count()
unique_days = silver_df.select("rate_date").distinct().count()
stale_count = silver_df.filter(col("is_stale") == True).count()



# Get the latest INR rate specifically
latest_inr =  ( silver_df
    .filter((col("currency") == "INR") & (col("base_code") == "USD"))
    .orderBy(F.desc("rate"))
    .select("rate_date", "rate")
    .limit(1)
    .collect()

)


print(f"\n[DONE] silver/exchange_rates/ written: {total} rows")
print(f"    Unique currencies {unique_curr}")
print(f"    Days of data: {unique_days}")
print(f"    Stale records: {stale_count}")

if latest_inr:
    latest_date = latest_inr[0]["rate_date"]
    latest_rate = latest_inr[0]["rate"]

    print(f" Latest USD→INR rate: {latest_rate}")
    print(f"Rate date: {latest_date}")
else:
    print(f" [WARN]:No USD-to-INR rate found")

print(f" \n[CURRENCY SUMMARY] Latest rates (most recent day) ")
silver_df\
    .filter(col("rate_date") == silver_df.agg(F.max("rate_date")).collect()[0][0])\
        .select("currency","rate", "rate_date")\
            .orderBy("currency")\
                .show(20)
    

display(silver_df.select(
    "base_code", "currency", "rate",
    "rate_date", "rate_year", "rate_month",
    "is_weekend", "is_stale", "_silver_load_ts")
        .orderBy(F.desc("rate_date"),"currency").limit(5)
)
    




[DONE] Read rows : 1328 rows
  (Expected: 166 currencies X number of days)
 [CLEAN] After null/zero removal: 1328 rows
[OK] Date parsing successful
[FILTER] After keeping relevant 10 currencies:80 

[DONE] silver/exchange_rates/ written: 80 rows
    Unique currencies 10
    Days of data: 8
    Stale records: 80
 Latest USD→INR rate: 95.46
Rate date: 2026-07-07
 
[CURRENCY SUMMARY] Latest rates (most recent day) 
+--------+------+----------+
|currency|  rate| rate_date|
+--------+------+----------+
|     AED|  3.67|2026-07-07|
|     AUD|  1.44|2026-07-07|
|     CAD|  1.42|2026-07-07|
|     CNY|  6.80|2026-07-07|
|     EUR|  0.87|2026-07-07|
|     GBP|  0.75|2026-07-07|
|     INR| 95.46|2026-07-07|
|     JPY|162.13|2026-07-07|
|     SGD|  1.29|2026-07-07|
|     USD|  1.00|2026-07-07|
+--------+------+----------+



base_code,currency,rate,rate_date,rate_year,rate_month,is_weekend,is_stale,_silver_load_ts
USD,AED,3.67,2026-07-07,2026,7,false,true,2026-07-11T19:55:41.546497Z
USD,AUD,1.44,2026-07-07,2026,7,false,true,2026-07-11T19:55:41.546497Z
USD,CAD,1.42,2026-07-07,2026,7,false,true,2026-07-11T19:55:41.546497Z
USD,CNY,6.80,2026-07-07,2026,7,false,true,2026-07-11T19:55:41.546497Z
USD,EUR,0.87,2026-07-07,2026,7,false,true,2026-07-11T19:55:41.546497Z
